# Bronze Layer v2 — Hourly Scheduled Load: Source Blob → Charging Sessions

**Day 3 | Scheduled via Databricks Job — runs every hour automatically.**

Reads the current hour's CSV files from the instructor-provided source blob and copies them into the Bronze Volume, preserving the exact source directory structure. No transformation — files land in Bronze exactly as they are in the source.

---

### Source layout
```
wasbs://source@dataenggdailystorage.blob.core.windows.net/
  └── realtime/
        └── charging_sessions/
              └── YYYY/MM/DD/HH/
                    └── sessions_YYYYMMDD_HHMM.csv
```

### Bronze target (mirrors source exactly)
```
/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/
  └── realtime/
        └── charging_sessions/
              └── YYYY/MM/DD/HH/
                    └── sessions_YYYYMMDD_HHMM.csv
```

---

### What changed from v1 → v2

| | v1 | v2 |
|---|---|---|
| **Partition selection** | Manually set `LOAD_YEAR`, `LOAD_MONTH`, `LOAD_DAY`, `LOAD_HOUR` in Cell 2 | Auto-computed from `datetime.now(UTC)` at runtime |
| **Trigger** | Run notebook manually | Databricks Job — cron `0 * * * *` (top of every hour) |
| **Full load** | `LOAD_MODE = "full"` | `FULL_LOAD_OVERRIDE = True` (one-off flag, reset to False after) |
| **Missing hour** | Crashes with `Path does not exist` | Checks folder exists first — exits cleanly with a warning |
| **Failure alerting** | None | Job marks run Failed + sends email if copy errors occur |

## Cell 1 — Authenticate to source blob

Reads the storage account name, container name, and SAS token from Key Vault via the `kv-ev-scope` Databricks secret scope. Nothing is hardcoded.

**Must run every session** — `spark.conf.set` registers the SAS credential for the `wasbs://` driver. This config does not persist across cluster restarts, so always run this cell first.

- `source-storage-account` → blob storage account name
- `source-container` → container name (`source`)
- `source-sas-token` → SAS token with `sp=rl` (read + list) permissions
- `SOURCE_ROOT` → base `wasbs://` URL used by all subsequent cells

In [ ]:
from datetime import datetime, timezone

SCOPE = "kv-ev-scope"

STORAGE_ACCOUNT = dbutils.secrets.get(scope=SCOPE, key="source-storage-account")
CONTAINER       = dbutils.secrets.get(scope=SCOPE, key="source-container")
SAS_TOKEN       = dbutils.secrets.get(scope=SCOPE, key="source-sas-token")

spark.conf.set(
    f"fs.azure.sas.{CONTAINER}.{STORAGE_ACCOUNT}.blob.core.windows.net",
    SAS_TOKEN
)

SOURCE_ROOT = f"wasbs://{CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net"

print(f"Storage account : {STORAGE_ACCOUNT}")
print(f"Container       : {CONTAINER}")
print(f"SAS token       : [REDACTED]")
print(f"Source root     : {SOURCE_ROOT}")
print("Source blob authenticated — OK")

## Cell 2 — Resolve load partition from system clock

**This cell never needs manual editing when running as a scheduled Job.**

`datetime.now(timezone.utc)` captures the exact UTC time when the Job fires. The Job cron `0 * * * *` fires at the top of every hour (09:00, 10:00, etc.), so `LOAD_HOUR` always matches the folder that the source system just finished writing for that hour.

Zero-padding is applied automatically via `strftime` — `6` becomes `"06"`, `9` becomes `"09"`, etc.

**`FULL_LOAD_OVERRIDE`** — set to `True` only when you need to re-copy all historical data in one shot (first run, disaster recovery, backfill). After the full load completes, set it back to `False` before the next scheduled run.

| `FULL_LOAD_OVERRIDE` | What gets copied |
|---|---|
| `False` (default) | Only the current hour folder — e.g. `2026/07/06/09/` |
| `True` | Everything under `realtime/charging_sessions/` — all years, months, days, hours |

In [ ]:
# Set to True only for a one-off full historical load.
# Leave False for all normal hourly scheduled runs.
FULL_LOAD_OVERRIDE = False

now = datetime.now(timezone.utc)

LOAD_YEAR  = now.strftime("%Y")   # e.g. "2026"
LOAD_MONTH = now.strftime("%m")   # e.g. "07"
LOAD_DAY   = now.strftime("%d")   # e.g. "06"
LOAD_HOUR  = now.strftime("%H")   # e.g. "09"

BRONZE_VOLUME = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume"
BASE_SUBPATH  = "realtime/charging_sessions"

if FULL_LOAD_OVERRIDE:
    source_path = f"{SOURCE_ROOT}/{BASE_SUBPATH}/"
    bronze_path = f"{BRONZE_VOLUME}/{BASE_SUBPATH}/"
    load_label  = "FULL (override)"
else:
    partition   = f"{LOAD_YEAR}/{LOAD_MONTH}/{LOAD_DAY}/{LOAD_HOUR}"
    source_path = f"{SOURCE_ROOT}/{BASE_SUBPATH}/{partition}/"
    bronze_path = f"{BRONZE_VOLUME}/{BASE_SUBPATH}/{partition}/"
    load_label  = f"INCREMENTAL — {partition}"

print(f"Run time (UTC)  : {now.strftime('%Y-%m-%d %H:%M:%S')} UTC")
print(f"Load mode       : {load_label}")
print(f"Source path     : {source_path}")
print(f"Bronze path     : {bronze_path}")

## Cell 3 — Check source folder exists

Before attempting to list or copy files, verify that the source hour folder actually exists in blob storage.

**Why this matters:** The source system may write the CSV a few minutes after the hour boundary. If the Job fires at exactly 09:00 UTC and the file arrives at 09:03, the copy would fail without this check.

**Behaviour:**
- Folder found → continues to Cell 4
- Folder not found → prints a warning and calls `dbutils.notebook.exit()` — the notebook stops here, the Job run is marked **Succeeded** (not Failed), and no alert email is sent

> If you want the Job to alert on missing hours (e.g. source system is down), change `dbutils.notebook.exit(msg)` to `raise Exception(msg)` — the Job will then mark the run as Failed and send the alert email.

In [ ]:
def folder_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

if not folder_exists(source_path):
    msg = f"Source folder not found — {source_path}. Data may not have arrived yet. Exiting."
    print(f"WARNING: {msg}")
    dbutils.notebook.exit(msg)

print(f"Source folder confirmed — {source_path}")

## Cell 4 — List source files

Recursively lists all files under the resolved source path.

- For a **scheduled hourly run** this is typically one CSV file: `sessions_YYYYMMDD_HHMM.csv`
- For a **full load override** this recurses through all year/month/day/hour subdirectories and returns every CSV

`list_files_recursive` walks into subdirectories by checking `item.isDir()`. Files are collected in a flat list — the relative path from `source_path` is used later in Cell 5 to reconstruct the destination path.

In [ ]:
def list_files_recursive(path):
    items = dbutils.fs.ls(path)
    files = []
    for item in items:
        if item.isDir():
            files.extend(list_files_recursive(item.path))
        else:
            files.append(item)
    return files

source_files = list_files_recursive(source_path)

if not source_files:
    msg = f"No files found at source path — {source_path}. Exiting."
    print(f"WARNING: {msg}")
    dbutils.notebook.exit(msg)

print(f"Files found: {len(source_files)}")
for f in source_files:
    size_kb = round(f.size / 1024, 1)
    print(f"  {f.path}  [{size_kb} KB]")

## Cell 5 — Copy files to Bronze Volume

Copies each source file to the Bronze Volume, recreating the exact directory structure.

**How the path is built:**
```
source_file.path  = wasbs://.../realtime/charging_sessions/2026/07/06/09/sessions_20260706_0900.csv
source_path       = wasbs://.../realtime/charging_sessions/2026/07/06/09/
relative_path     = sessions_20260706_0900.csv          ← strip source_path prefix
dest_path         = /Volumes/.../2026/07/06/09/sessions_20260706_0900.csv  ← bronze_path + relative
```

**Idempotency:** `dbutils.fs.cp` overwrites any existing file at the destination — re-running the same hour always produces the same result with no duplicate risk.

**On failure:** if any file fails to copy, it is added to `skipped` and logged. After all files are attempted, a `raise Exception` fires — the Job marks the run as **Failed** and sends an alert email.

In [ ]:
copied  = []
skipped = []

for file_info in source_files:
    relative_path = file_info.path.replace(source_path, "")
    dest_path     = bronze_path + relative_path

    try:
        dbutils.fs.cp(file_info.path, dest_path)
        copied.append(dest_path)
        print(f"  COPIED  {relative_path}")
    except Exception as e:
        skipped.append((file_info.path, str(e)))
        print(f"  FAILED  {relative_path} — {e}")

print(f"\nCopy complete: {len(copied)} copied, {len(skipped)} failed")

if skipped:
    raise Exception(f"{len(skipped)} file(s) failed to copy — check output above.")

## Cell 6 — Verify files in Bronze Volume

Lists the files now present at `bronze_path` and asserts the count matches the source file count.

If the assertion fails (e.g. a partial copy where some files succeeded and some silently skipped), the Job run is marked **Failed** and an alert fires.

Compare this output against Cell 4 — every file listed in source should appear here with the same filename.

In [ ]:
bronze_files = list_files_recursive(bronze_path)

print(f"Files in Bronze Volume: {len(bronze_files)}")
for f in bronze_files:
    size_kb = round(f.size / 1024, 1)
    print(f"  {f.path}  [{size_kb} KB]")

assert len(bronze_files) == len(source_files), (
    f"File count mismatch — source: {len(source_files)}, bronze: {len(bronze_files)}"
)
print("Verification passed — source and Bronze file counts match.")

## Cell 7 — Read sample file and inspect schema

Reads the first copied CSV from the Bronze Volume into a Spark DataFrame and prints the inferred schema and 5 sample rows.

- `header=True` — uses the first CSV row as column names
- `inferSchema=True` — Spark detects column types (string, int, timestamp, etc.)

**This is a lightweight sanity check.** The Silver layer will enforce an explicit schema — `inferSchema` is used here only to confirm the file is readable and the columns look as expected. All types at Bronze are acceptable as strings.

In [ ]:
sample_file = bronze_files[0].path
print(f"Reading sample: {sample_file}")

df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(sample_file)

print(f"Row count : {df.count():,}")
print(f"Columns   : {len(df.columns)}")
df.printSchema()
display(df.limit(5))

## Cell 8 — Run summary

Prints a final summary of the run: timestamp, load mode, paths, file counts, and any failures.

When running as a scheduled Job, this output is visible in:
**Databricks → Workflows → `job_bronze_charging_sessions_hourly` → Run history → this run → Output**

---

**What comes next:** This notebook lands raw CSVs in the Bronze Volume. The Silver layer notebook (Day 7) reads from `/Volumes/.../bronze-volume/realtime/charging_sessions/`, applies explicit schema, deduplicates by `session_id`, and writes Delta.

In [ ]:
print("=" * 60)
print("BRONZE BLOB MIGRATION v2 — HOURLY RUN SUMMARY")
print("=" * 60)
print(f"Run time (UTC)  : {now.strftime('%Y-%m-%d %H:%M:%S')} UTC")
print(f"Load mode       : {load_label}")
print(f"Source path     : {source_path}")
print(f"Bronze path     : {bronze_path}")
print(f"Files copied    : {len(copied)}")
print(f"Files failed    : {len(skipped)}")
if skipped:
    print("\nFailed files:")
    for src, err in skipped:
        print(f"  {src}  →  {err}")
print("=" * 60)
print("Next step: Silver layer reads from Bronze Volume and writes Delta.")